Flower classification using CNN

libraries 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

load the data 

In [ ]:
train_dir = '/content/drive/MyDrive/colab projects/Rakshitha - Dissertation/train/'
test_dir = '/content/drive/MyDrive/colab projects/Rakshitha - Dissertation/test/'

Defining Image Size and Batch size for Data Generators

In [ ]:
img_size = (128, 128)
batch_size = 32

Data visualisation 

In [ ]:
# Load the dataset
data_dir = '/content/drive/MyDrive/colab projects/Rakshitha - Dissertation/train/'
flowers = ['daisy', 'dandelion', 'rose', 'sunflower', 'tulip']
data = []
for flower in flowers:
    flower_dir = os.path.join(data_dir, flower)
    for img in os.listdir(flower_dir):
        data.append((os.path.join(flower_dir, img), flower))

df = pd.DataFrame(data, columns=['image', 'label'])

# Explore the dataset
print("Shape of dataset:", df.shape)
print(df.describe())
print(df.head())
print(df.tail())

# Visualize the dataset
plt.figure(figsize=(8,6))
sns.countplot(df['label'])
plt.title("Distribution of Flowers")
plt.show()

plt.figure(figsize=(8,6))
df['label'].value_counts().plot(kind='pie', autopct='%1.1f%%')
plt.title("Percentage Distribution of Flowers")
plt.show()

plt.figure(figsize=(8,6))
df['label'].value_counts().plot(kind='bar')
plt.title("Count of Flowers")
plt.show()

# Display some sample images
fig, axs = plt.subplots(3, 3, figsize=(12,12))
axs = axs.flatten()
for i in range(9):
    img = plt.imread(df['image'][i])
    axs[i].imshow(img)
    axs[i].set_title(df['label'][i])
plt.show()

image generatior 

In [ ]:
train_datagen = ImageDataGenerator(rescale=1./255, 
                                   shear_range=0.2, 
                                   zoom_range=0.2, 
                                   horizontal_flip=True, 
                                   validation_split=0.2)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(train_dir, 
                                                    target_size=img_size, 
                                                    batch_size=batch_size, 
                                                    class_mode='categorical', 
                                                    subset='training')

valid_generator = train_datagen.flow_from_directory(train_dir, 
                                                    target_size=img_size, 
                                                    batch_size=batch_size, 
                                                    class_mode='categorical', 
                                                    subset='validation')

test_generator = test_datagen.flow_from_directory(test_dir, 
                                                  target_size=img_size, 
                                                  batch_size=1, 
                                                  class_mode=None, 
                                                  shuffle=False)


Develop a CNN Sequential model

In [ ]:
model = Sequential()

model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(img_size[0], img_size[1], 3)))
model.add(MaxPooling2D((2, 2)))

model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))

model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))

model.add(Flatten())

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(5, activation='softmax'))

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
# Fit the model to the training data
es = EarlyStopping(monitor='val_loss', patience=3)
history = model.fit(train_generator, epochs=100, callbacks=[es])

In [ ]:
# Print the accuracy and F1 score on the training data
train_loss, train_accuracy = model.evaluate(train_generator, verbose=0)
train_f1_score = 2 * (train_accuracy * train_loss) / (train_accuracy + train_loss)
print('Training accuracy:', train_accuracy)
print('Training F1 score:', train_f1_score)